In [25]:

import pyrosetta
from benchmark import bpti_ryfyn_benchmark
from rotamer_extraction import extract_top_n_rotamers
from h_ising_creation import extract_hamiltonian_tensors, build_ising_hamiltonian, reduce_hamiltonian
from initialisation import initialize_rosetta

import numpy as np
import pennylane as qml

initialize_rosetta(pyrosetta, extra_flags="-mute all")

Initializing PyRosetta with cleaning flags: -ignore_unrecognized_res and extra flags: -mute all
┌───────────────────────────────────────────────────────────────────────────────┐
│                                  PyRosetta-4                                  │
│               Created in JHU by Sergey Lyskov and PyRosetta Team              │
│               (C) Copyright Rosetta Commons Member Institutions               │
│                                                                               │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRES PURCHASE OF A LICENSE │
│          See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└───────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2026 [Rosetta PyRosetta4.Release.python312.ubuntu 2026.03+releasequarterly.5e498f1409c68ade56c8ce5842bf79e1b02e8db4 2026-01-13T13:24:11] retrieved from: http://www.pyrosetta.org


In [26]:
benchmark_pose = bpti_ryfyn_benchmark()

results = extract_top_n_rotamers(benchmark_pose)
pyrosetta.rosetta.core.io.pdb.dump_pdb(benchmark_pose, "benchmark_pose.pdb")

Fragment Sequence: RYFYN
Total Residues: 5
Creating score function
Pose scored successfully!
Creating Repacking Task - Core Rotamer Optimisation Protocol
Computing One-Body and Two-Body Energies
Iterating through molten residues - determining the top rotamer positions for each amino acid
1 1
2 2
3 3
4 4
5 5


True

In [27]:
rotamer_lib, ig, rot_sets = results
h_linear, J_quadratic = extract_hamiltonian_tensors(rotamer_lib, ig, rot_sets)


In [28]:
h_flex_linear, J_flex_quadratic, global_offset = reduce_hamiltonian(h_linear, J_quadratic, rotamer_lib)
for idx in h_linear:
    print(h_linear[idx])
    print(h_flex_linear.get(idx, "None"))
    print("\n---------------------------------------\n")

{0: 1.2480376958847046, 1: 1.5509175062179565, 2: 1.6199729442596436, 3: 1.8183133602142334}
{0: 1.2480376958847046, 1: 1.5509175062179565, 2: 1.6199729442596436, 3: 1.8183133602142334}

---------------------------------------

{0: 2.0819952487945557, 1: 2.081996440887451, 2: 3.0249295234680176, 3: 3.024930477142334}
{0: 2.0819952487945557, 1: 2.081996440887451, 2: 3.0249295234680176, 3: 3.024930477142334}

---------------------------------------

{0: 2.783350706100464, 1: 3.331935405731201, 2: 4.205162048339844}
{0: 2.783350706100464, 1: 3.331935405731201, 2: 4.205162048339844}

---------------------------------------

{0: 2.8021299839019775, 1: 2.802131175994873, 2: 3.082185983657837, 3: 3.0821871757507324}
{0: 2.8021299839019775, 1: 2.802131175994873, 2: 3.082185983657837, 3: 3.0821871757507324}

---------------------------------------

{0: 0.20771783590316772, 1: 0.33269232511520386, 2: 0.6052141785621643, 3: 1.415124535560608}
{0: 0.20771783590316772, 1: 0.33269232511520386, 2: 0.

In [29]:
hamiltonian = build_ising_hamiltonian(h_flex_linear, J_flex_quadratic, global_offset, penalty=0.0)

Reduced Hamiltonian built: 19 Qubits, 116 Pauli strings.


In [30]:
print(hamiltonian)

384.384590045549 * I(0) + -8.58522180840373 * Z(0) + -7.591564707458019 * Z(1) + -7.588681485503912 * Z(2) + -7.920536927878857 * Z(3) + -8.084025613963604 * Z(4) + -8.084026210010052 * Z(5) + -8.62156843394041 * Z(6) + -8.621568910777569 * Z(7) + 2.0934013416990638 * Z(8) + -342.9867805019021 * Z(9) + 1.0110462680459023 * Z(10) + 0.23341885954141617 * Z(11) + 0.26594097167253494 * Z(12) + -0.5092299804091454 * Z(13) + -0.5092305764555931 * Z(14) + -95.42025848478079 * Z(15) + -84.68211737647653 * Z(16) + -70.51118155568838 * Z(17) + -92.10653268266469 * Z(18) + 2.239682912826538 * (Z(0) @ Z(4)) + 2.239682912826538 * (Z(0) @ Z(5)) + 2.2561678886413574 * (Z(0) @ Z(6)) + 2.2561678886413574 * (Z(0) @ Z(7)) + -0.5157089829444885 * (Z(0) @ Z(8)) + -0.04511748626828194 * (Z(0) @ Z(9)) + -0.46967217326164246 * (Z(0) @ Z(10)) + 1.870367407798767 * (Z(1) @ Z(4)) + 1.870367407798767 * (Z(1) @ Z(5)) + 1.8839359283447266 * (Z(1) @ Z(6)) + 1.8839359283447266 * (Z(1) @ Z(7)) + -0.3614487051963806 * 

In [52]:
import pennylane as qml
from pennylane import numpy as np

H_ising = hamiltonian
# Assuming 'H_ising' is the Hamiltonian returned by your build_ising_hamiltonian function
num_qubits = 19
dev = qml.device('lightning.gpu', wires=range(num_qubits))

# Define the standard X-mixer for QAOA
mixer_h = qml.dot([1.0] * num_qubits, [qml.PauliX(i) for i in range(num_qubits)])
# THE STANDARD MIXER DIDN'T WORK, CUSTOM MIXER

seq_positions = sorted(list(h_flex_linear.keys()))
wire_offsets = {}
current_wire = 0
for seq in seq_positions:
    wire_offsets[seq] = current_wire
    current_wire += len(h_flex_linear[seq])

def build_xy_mixer(wire_offsets, h_flex):
    """
    Constructs the XY mixer Hamiltonian, restricting mixing strictly
    to intra-residue rotamer wires.
    """
    coeffs = []
    observables = []

    seq_positions = sorted(list(h_flex.keys()))
    for seq in seq_positions:
        base_wire = wire_offsets[seq]
        num_rots = len(h_flex[seq])

        # Apply XY mixing between all pairs of rotamers within THIS residue
        for i in range(num_rots):
            for j in range(i + 1, num_rots):
                w_i = base_wire + i
                w_j = base_wire + j

                # 0.5 * (X_i X_j + Y_i Y_j)
                coeffs.extend([0.5, 0.5])
                observables.extend([
                    qml.PauliX(w_i) @ qml.PauliX(w_j),
                    qml.PauliY(w_i) @ qml.PauliY(w_j)
                ])

    return qml.Hamiltonian(coeffs, observables)

# Instantiate the custom mixer
H_mixer_xy = build_xy_mixer(wire_offsets, h_flex_linear)

def custom_xy_mixer_layer(beta, h_flex, wire_offsets):
    seq_positions = sorted(list(h_flex.keys()))

    for seq in seq_positions:
        base_wire = wire_offsets[seq]
        num_rots = len(h_flex[seq])

        for i in range(num_rots):
            for j in range(i + 1, num_rots):
                w_i = base_wire + i
                w_j = base_wire + j

                qml.IsingXY(beta, wires=[w_i, w_j])

def qaoa_layer(gamma, beta):
    qml.qaoa.cost_layer(gamma, H_ising)
    custom_xy_mixer_layer(beta, h_flex_linear, wire_offsets)

@qml.qnode(dev)
def cost_function(params):
    # params is a 2D array of shape (2, p) where p is the number of QAOA layers
    gammas = params[0]
    betas = params[1]

    # # 1. Initialize in a superposition state
    # for i in range(num_qubits):
    #     qml.Hadamard(wires=i)

    # Custom initialisation
    for seq in seq_positions:
        # print("Flipping", wire_offsets[seq])
        base_wire = wire_offsets[seq]
        qml.PauliX(wires=base_wire) # First rotamer set to 1

    # 2. Apply p layers of QAOA
    for i in range(len(gammas)):
        qaoa_layer(gammas[i], betas[i])

    # 3. Measure the expectation value of the cost Hamiltonian
    return qml.expval(H_ising)

# Define depth 'p'. Start small (p=2) on your M1 to gauge execution time before scaling.
p = 8
np.random.seed(42)
# Initialize parameters close to zero to avoid barren plateaus
initial_params = np.random.uniform(low=-0.01, high=0.01, size=(2, p), requires_grad=True)

In [53]:

# Optimization Execution
opt = qml.AdamOptimizer(stepsize=0.01)
epochs = 150
current_params = initial_params
lowest_param_set = (float('inf'), current_params)

print("Commencing QAOA Optimization...")
for epoch in range(epochs):
    current_params, cost = opt.step_and_cost(cost_function, current_params)
    if cost < lowest_param_set[0]:
        lowest_param_set = (cost, current_params)
    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | Cost: {cost:.4f}")

print("Optimization converged.")

Commencing QAOA Optimization...
Epoch   0 | Cost: 13.4258
Epoch  10 | Cost: 13.4821
Epoch  20 | Cost: 13.5427
Epoch  30 | Cost: 13.5148
Epoch  40 | Cost: 13.4422
Epoch  50 | Cost: 13.4252
Epoch  60 | Cost: 13.4202
Epoch  70 | Cost: 13.4179
Epoch  80 | Cost: 13.4177
Epoch  90 | Cost: 13.4177
Epoch 100 | Cost: 13.4177
Epoch 110 | Cost: 13.4177
Epoch 120 | Cost: 13.4176
Epoch 130 | Cost: 13.4176
Epoch 140 | Cost: 13.4176
Optimization converged.


In [59]:
@qml.qnode(dev)
def sample_function(params):
    for i in range(num_qubits):
        qml.Hadamard(wires=i)
    for i in range(len(params[0])):
        qaoa_layer(params[0][i], params[1][i])
    return qml.probs(wires=range(num_qubits))

@qml.qnode(dev)
def new_sample_function(params):
    for seq in seq_positions:
        base_wire = wire_offsets[seq]
        qml.PauliX(wires=base_wire)

    for i in range(len(params[0])):
        qaoa_layer(params[0][i], params[1][i])

    return qml.probs(wires=range(num_qubits))

# Generate the probability vector for all 2^19 states
print(f"Lowest parameter cost: {lowest_param_set[0]}")
probabilities = new_sample_function(lowest_param_set[1])
# probabilities = new_sample_function(current_params)

Lowest parameter cost: 13.417628393556443


In [60]:
top_k = 100
top_indices = np.argsort(probabilities)[-top_k:][::-1]
top_indices

tensor([279688, 278920, 279681, 265352, 267400, 279576, 279592, 279682,
         50312, 271496, 279624, 279684, 148616,  83080, 279170, 279172,
        279176, 279169,  49800, 148104, 278914,  82568, 278916, 278913,
         49544,  82312, 147848, 264584, 266632, 278808, 278824, 265345,
        267393, 279569, 279585, 265240, 267288, 265256, 267304,  35976,
         38024, 270728, 134280, 136328, 278856,  68744,  70792, 265346,
        267394,  50305, 271489, 279617, 279570, 279586, 265288, 267336,
        265348, 267396,  50200,  50216, 271384, 271400, 279572, 279588,
        148609,  83073, 264834, 266882, 148504, 148520,  50306,  82968,
         82984, 271490, 279058, 279074, 279618,  42120,  50248, 271432,
         50308, 271492, 279620, 148610,  83074, 140424, 148552,  74888,
        270978,  83016, 264836, 266884, 148612, 279106,  83076, 148098,
        279060, 279076,  49794, 264840], requires_grad=True)

In [64]:

# 1. Extract the top N most probable bitstrings
top_k = 100
# np.argsort returns indices; we take the last 'top_k' and reverse them for descending order
top_indices = list(np.argsort(probabilities)[-top_k:][::-1])
# top_indices.append(279688) # first 1000 1000 100 1000 1000

valid_conformations = []

# Helper to convert integer index back to binary list (length = 19)
def int_to_bitstring(idx, length):
    return [int(x) for x in format(idx, f'0{length}b')]

# 2. Enforce the One-Hot Constraint
for idx in top_indices:
    bitstring = int_to_bitstring(idx, num_qubits)
    is_valid = True

    # Iterate through each residue's allocated wires using your existing `wire_offsets`
    # and the known length of h_flex[seq]
    for seq in seq_positions:
        start_wire = wire_offsets[seq]
        num_rots = len(h_flex_linear[seq])

        # Sum the bits corresponding to this residue's rotamers
        residue_sum = sum(bitstring[start_wire : start_wire + num_rots])

        if residue_sum != 1:
            is_valid = False
            break # Fails the penalty constraint

    if is_valid:
        # 3. Calculate True Biological Energy (Classical PyRosetta Equation)
        # using the valid bitstring against the original h_flex and J_flex tensors.
        bio_energy = 0 # calculate_classical_energy(bitstring, h_flex, J_flex, global_offset)
        valid_conformations.append({
            "bitstring": bitstring,
            "probability": probabilities[idx],
            "energy": bio_energy
        })
print(wire_offsets)
if not valid_conformations:
    raise ValueError("Zero valid conformations found in the top sampled states. You must increase QAOA depth 'p' or increase the penalty multiplier.")

# Sort the strictly valid conformations by their true biological energy
valid_conformations.sort(key=lambda x: x['energy'])
best_conformation = valid_conformations[0]

print(f"Optimal Valid Sequence: {best_conformation['bitstring']}")
print(f"Classical Energy: {best_conformation['energy']} kcal/mol")
print(f"Valid to Non-Valid Ration: {len(valid_conformations)} - {len(top_indices) - len(valid_conformations)}")

{1: 0, 2: 4, 3: 8, 4: 11, 5: 15}
Optimal Valid Sequence: [1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0]
Classical Energy: 0 kcal/mol
Valid to Non-Valid Ration: 100 - 0
